In [ ]:
import numpy as np
import pandas as pd
import torch
import json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# -------------------------- 定义常量 --------------------------
# -------------------------- 定义常量 --------------------------
hidden_states = [
    "Recon", "InitialAccess", "Exploit", "PrivilegeEscalation", 
    "DefenseEvasion", "CredentialAccess", "Discovery", "LateralMovement",
    "Persistence", "CommandAndControl", "Collection", "Exfiltration",
    "Impact", "ResourceDevelopment", "Execution", "DataStaging",
    "InternalRecon", "DataEncryption", "SystemManipulation", "Cleanup"
]

event_types = [
    "NetworkScan", "VulnScan", "Malware", "DataTransfer",
    "PortScan", "BruteForce", "Backdoor", "RootKit", 
    "CredentialDumping", "KeyLogging", "ProcessInjection", "ServiceCreation",
    "RegistryModification", "FileAccess", "DatabaseAccess", "EmailExfil",
    "DomainEnumeration", "PowerShellExecution", "ScriptExecution", "LogDeletion",
    "SystemManipulation", "DataEncryption"
]
# 创建映射字典
event_to_code = {event: idx for idx, event in enumerate(event_types)}
state_to_code = {state: idx for idx, state in enumerate(hidden_states)}


# -------------------------- 数据预处理 --------------------------
class AttackSequenceDataset(Dataset):
    def __init__(self, events, states, seq_length=5):
        self.seq_length = seq_length
        
        # One-hot编码事件和状态
        # 修改sparse为sparse_output
        self.event_encoder = OneHotEncoder(sparse_output=False)
        self.state_encoder = OneHotEncoder(sparse_output=False)
        
        events_reshaped = np.array(events).reshape(-1, 1)
        states_reshaped = np.array(states).reshape(-1, 1)
        
        self.event_encoded = self.event_encoder.fit_transform(events_reshaped)
        self.state_encoded = self.state_encoder.fit_transform(states_reshaped)
        
        # 创建序列
        self.sequences = []
        self.labels = []
        
        for i in range(len(events) - seq_length):
            self.sequences.append(self.event_encoded[i:i+seq_length])
            self.labels.append(self.state_encoded[i+seq_length])
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.FloatTensor(self.labels[idx])


# -------------------------- 定义LSTM模型 --------------------------
class AttackStageLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(AttackStageLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        # LSTM前向传播
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # 只使用最后一个时间步的输出
        out = self.softmax(out)
        return out

# -------------------------- 训练模型 --------------------------
def train_model(model, train_loader, criterion, optimizer, num_epochs=50):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')


# -------------------------- 加载数据集 --------------------------
def load_dataset(filename):
    """
    从JSON文件加载攻击序列数据
    
    Args:
        filename: JSON文件名
        
    Returns:
        df: 包含states和events的DataFrame
        sequence_lengths: 每个序列的长度列表
    """
    with open(filename, 'r') as f:
        sequences = json.load(f)
        
    all_states = []
    all_events = []
    sequence_lengths = []
    
    for seq in sequences:
        all_states.extend(seq['states'])
        all_events.extend(seq['events'])
        sequence_lengths.append(len(seq['states']))
        
    df = pd.DataFrame({
        'state': all_states,
        'event': all_events
    })
    
    return df, sequence_lengths



# -------------------------- 主程序 --------------------------
if __name__ == "__main__":
    # 加载数据
    print("加载数据...")
    df, sequence_lengths = load_dataset('attack_sequences.json')
    
    # 准备数据集
    seq_length = 5  # 序列长度
    dataset = AttackSequenceDataset(df['event'].values, df['state'].values, seq_length)
    
    # 划分训练集和测试集
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    
    # 创建数据加载器
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # 初始化模型
    input_size = len(event_types)  # 输入维度（事件的one-hot编码维度）
    hidden_size = 64  # LSTM隐藏层大小
    num_layers = 2   # LSTM层数
    output_size = len(hidden_states)  # 输出维度（状态的数量）
    
    model = AttackStageLSTM(input_size, hidden_size, num_layers, output_size)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    # 训练模型
    train_model(model, train_loader, criterion, optimizer)
    
    # 测试模型
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            outputs = model(batch_x)
            _, predicted = torch.max(outputs.data, 1)
            _, actual = torch.max(batch_y.data, 1)
            total += batch_y.size(0)
            correct += (predicted == actual).sum().item()
    
    print(f'测试集准确率: {100 * correct / total:.2f}%')
    
    # 示例预测
    sample_sequence = df['event'].values[:seq_length]
    sample_sequence_encoded = dataset.event_encoder.transform(np.array(sample_sequence).reshape(-1, 1))
    sample_sequence_tensor = torch.FloatTensor(sample_sequence_encoded).unsqueeze(0)
    
    with torch.no_grad():
        prediction = model(sample_sequence_tensor)
        predicted_state_idx = torch.argmax(prediction).item()
        predicted_state = list(state_to_code.keys())[predicted_state_idx]
        
    print(f"\n示例预测:")
    print(f"输入序列: {sample_sequence}")
    print(f"预测的下一个攻击阶段: {predicted_state}")

In [16]:
import numpy as np
import pandas as pd
import torch
import json
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# -------------------------- 定义常量 --------------------------
hidden_states = ['CAPEC'+ str(i) for i in range(1,500)]
event_types = ['CAR'+ str(i) for i in range(1,1000)]  # 修改为1000以获得999维输入
# 创建映射字典
event_to_code = {event: idx for idx, event in enumerate(event_types)}
state_to_code = {state: idx for idx, state in enumerate(hidden_states)}


# -------------------------- 数据预处理 --------------------------
class AttackSequenceDataset(Dataset):
    def __init__(self, events, states, seq_length=5):
        self.seq_length = seq_length
        
        # One-hot编码事件和状态
        self.event_encoder = OneHotEncoder(sparse_output=False)
        self.state_encoder = OneHotEncoder(sparse_output=False)
        
        # 确保所有事件都在预定义的event_types中
        events_list = list(events)
        for event in events_list:
            if event not in event_to_code:
                # 如果发现未知事件，用'CAR1'替代
                events_list[events_list.index(event)] = 'CAR1'
                
        events_reshaped = np.array(events_list).reshape(-1, 1)
        states_reshaped = np.array(states).reshape(-1, 1)
        
        # 使用预定义的事件类型进行fit
        self.event_encoder.fit(np.array(event_types).reshape(-1, 1))
        self.state_encoder.fit(np.array(hidden_states).reshape(-1, 1))
        
        # 转换数据
        self.event_encoded = self.event_encoder.transform(events_reshaped)
        self.state_encoded = self.state_encoder.transform(states_reshaped)
        
        # 创建序列
        self.sequences = []
        self.labels = []
        
        for i in range(len(events_list) - seq_length):
            self.sequences.append(self.event_encoded[i:i+seq_length])
            # 修改标签格式 - 使用状态的索引而不是one-hot编码
            self.labels.append(np.where(self.state_encoded[i+seq_length] == 1)[0][0])
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.LongTensor([self.labels[idx]])[0]


# -------------------------- 改进的LSTM模型 --------------------------
class ImprovedAttackStageLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.3):
        super(ImprovedAttackStageLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # 添加更多层和dropout
        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True,
            dropout=dropout,
            bidirectional=True  # 使用双向LSTM
        )
        
        # 添加注意力机制
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        # 添加更多全连接层
        self.fc_layers = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size)
        )
        
    def attention_net(self, lstm_output):
        attention_weights = self.attention(lstm_output)
        attention_weights = torch.softmax(attention_weights, dim=1)
        context = torch.sum(attention_weights * lstm_output, dim=1)
        return context
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # 初始化隐藏状态
        h0 = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size).to(x.device)
        
        # LSTM前向传播
        lstm_out, _ = self.lstm(x, (h0, c0))
        
        # 应用注意力机制
        attn_out = self.attention_net(lstm_out)
        
        # 通过全连接层
        out = self.fc_layers(attn_out)
        return out

# -------------------------- 改进的训练函数 --------------------------
def train_model(model, train_loader, criterion, optimizer, num_epochs=100, scheduler=None):
    best_loss = float('inf')
    patience = 100
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            
            # 计算损失
            loss = criterion(outputs, batch_y)
            
            # 反向传播
            loss.backward()
            
            # 梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item()
            
            # 计算训练准确率
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        # 计算平均损失和准确率
        avg_loss = total_loss / len(train_loader)
        accuracy = 100 * correct / total
        
        # 学习率调整
        if scheduler:
            scheduler.step(avg_loss)  # 传入验证损失作为指标
        
        # 早停机制
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
            
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

# -------------------------- 主程序 --------------------------
if __name__ == "__main__":
    # 加载数据
    print("加载数据...")
    df, sequence_lengths = load_dataset('attack_sequences.json')
    
    # 准备数据集
    seq_length = 8  # 增加序列长度
    dataset = AttackSequenceDataset(df['event'].values, df['state'].values, seq_length)
    
    # 划分训练集和测试集
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    
    # 创建数据加载器
    batch_size = 64  # 增加batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # 初始化改进的模型
    input_size = len(event_types)  # 现在是999维
    hidden_size = 128  # 增加隐藏层大小
    num_layers = 3    # 增加层数
    output_size = len(hidden_states)
    
    model = ImprovedAttackStageLSTM(input_size, hidden_size, num_layers, output_size)
    
    # 使用权重初始化
    for name, param in model.named_parameters():
        if 'weight' in name:
            nn.init.xavier_normal_(param)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    
    # 训练模型
    train_model(model, train_loader, criterion, optimizer, num_epochs=100, scheduler=scheduler)

加载数据...


f:\python3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [10/100], Loss: 3.4047, Accuracy: 28.82%
Epoch [20/100], Loss: 2.7360, Accuracy: 33.98%
Epoch [30/100], Loss: 2.2954, Accuracy: 37.98%
Epoch [40/100], Loss: 1.9597, Accuracy: 42.20%
Epoch [50/100], Loss: 1.6989, Accuracy: 47.51%
Epoch [60/100], Loss: 1.4960, Accuracy: 51.67%
Epoch [70/100], Loss: 1.3248, Accuracy: 56.45%
Epoch [80/100], Loss: 1.1563, Accuracy: 61.20%
Epoch [90/100], Loss: 1.0193, Accuracy: 64.57%
Epoch [100/100], Loss: 0.8824, Accuracy: 69.74%


In [20]:
# 保存模型
print("保存模型...")
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'event_to_code': event_to_code,
    'state_to_code': state_to_code,
    'input_size': input_size,
    'hidden_size': hidden_size, 
    'num_layers': num_layers,
    'output_size': output_size
}, 'lstm_model.pth')
print("模型保存完成")


保存模型...
模型保存完成


In [21]:
print("加载模型...")
checkpoint = torch.load('lstm_model.pth')

# 重建模型架构
model = ImprovedAttackStageLSTM(
    checkpoint['input_size'],
    checkpoint['hidden_size'],
    checkpoint['num_layers'], 
    checkpoint['output_size']
)

# 加载模型参数
model.load_state_dict(checkpoint['model_state_dict'])

# 重建优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# 重建学习率调度器
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# 加载映射字典
event_to_code = checkpoint['event_to_code']
state_to_code = checkpoint['state_to_code']

print("模型加载完成")


加载模型...
模型加载完成


f:\python3\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [22]:
# 创建数据加载器
batch_size = 64  # 增加batch size

# 划分训练集、验证集和测试集
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# 定义评估函数
def evaluate_model(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = 100 * correct / total
    
    # 计算精确率
    from sklearn.metrics import precision_score, f1_score
    precision = precision_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return accuracy, precision, f1

accuracy, precision, f1 = evaluate_model(model, test_loader)
print(f"测试集准确率: {accuracy:.2f}%")
print(f"精确率: {precision:.4f}")
print(f"F1分数: {f1:.4f}")



测试集准确率: 72.02%
精确率: 0.7465
F1分数: 0.7010


f:\python3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
